# 📊 VitalNode — Visualización de datos de sesión
Sube tu archivo CSV generado por el sistema y ejecuta las celdas en orden.

In [ ]:
# ── 1. Subir el CSV desde tu ordenador ──────────────────────────────────────
from google.colab import files
import io
import pandas as pd

uploaded = files.upload()  # Se abrirá un selector de archivo
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Normalizar nombres de columnas (quitar espacios extra)
df.columns = df.columns.str.strip()

# Convertir timestamp de milisegundos a segundos
df['Time_s'] = df['Timestamp'] / 1000

# Convertir Posture_State a numérico (0 = recto, 1 = encorvado)
df['Posture_num'] = df['Posture_State'].astype(str).str.extract(r'(\d)').astype(float)

print(f'✅ CSV cargado: {len(df)} filas, {df["Time_s"].max():.0f}s de sesión')
df.head()

In [ ]:
# ── 2. Librerías y estilo ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Paleta de colores VitalNode (azul Deusto)
C_BPM     = '#185FA5'   # azul
C_HRV     = '#0F6E56'   # verde teal
C_RESP    = '#BA7517'   # ámbar
C_POSTURE = '#D85A30'   # coral
C_STRESS  = ['#639922', '#EF9F27', '#E24B4A']  # bajo / medio / alto

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F8F8',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})
print('✅ Librerías listas')

In [ ]:
# ── 3. Gráfica 1: BPM a lo largo de la sesión ────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(df['Time_s'], df['BPM'], color=C_BPM, linewidth=1.8, label='BPM')
ax.axhline(df['BPM'].mean(), color=C_BPM, linestyle='--', alpha=0.5,
           label=f'Media: {df["BPM"].mean():.1f} BPM')

# Rango clínico normal en reposo
ax.axhspan(60, 100, alpha=0.06, color=C_BPM, label='Rango normal (60-100)')

ax.set_xlabel('Tiempo (s)')
ax.set_ylabel('BPM')
ax.set_title('Frecuencia cardíaca durante la sesión', fontweight='bold')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('grafica_BPM.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  Mín: {df["BPM"].min():.1f}  Máx: {df["BPM"].max():.1f}  Media: {df["BPM"].mean():.1f} BPM')

In [ ]:
# ── 4. Gráfica 2: HRV Index (estrés) ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))

ax.fill_between(df['Time_s'], df['HRV_Index'], alpha=0.3, color=C_HRV)
ax.plot(df['Time_s'], df['HRV_Index'], color=C_HRV, linewidth=1.6, label='HRV Index')
ax.axhline(df['HRV_Index'].mean(), color=C_HRV, linestyle='--', alpha=0.6,
           label=f'Media: {df["HRV_Index"].mean():.1f}')

ax.set_xlabel('Tiempo (s)')
ax.set_ylabel('HRV Index')
ax.set_title('Variabilidad de la frecuencia cardíaca (HRV)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('grafica_HRV.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5. Gráfica 3: Respiración (FSR raw) ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(df['Time_s'], df['Respiration_Raw'], color=C_RESP, linewidth=1.4,
        label='FSR Raw (expansión torácica)')

# Detectar picos de inhalación (valores > percentil 75)
threshold = df['Respiration_Raw'].quantile(0.75)
peaks_mask = df['Respiration_Raw'] > threshold
ax.scatter(df.loc[peaks_mask, 'Time_s'], df.loc[peaks_mask, 'Respiration_Raw'],
           color=C_POSTURE, s=20, zorder=5, label='Pico de inhalación')

ax.set_xlabel('Tiempo (s)')
ax.set_ylabel('Valor raw FSR')
ax.set_title('Señal respiratoria (sensor de fuerza FSR)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('grafica_Respiracion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6. Gráfica 4: Estado de postura a lo largo del tiempo ────────────────────
fig, ax = plt.subplots(figsize=(12, 3))

# Colorear franjas de tiempo por estado de postura
for i in range(len(df) - 1):
    color = C_POSTURE if df['Posture_num'].iloc[i] == 1 else C_HRV
    ax.axvspan(df['Time_s'].iloc[i], df['Time_s'].iloc[i+1],
               alpha=0.6, color=color, linewidth=0)

recto   = mpatches.Patch(color=C_HRV,     label='Espalda recta (0)')
encorvado = mpatches.Patch(color=C_POSTURE, label='Espalda encorvada (1)')
ax.legend(handles=[recto, encorvado], loc='upper right')

pct_mal = df['Posture_num'].mean() * 100
ax.set_xlabel('Tiempo (s)')
ax.set_yticks([])
ax.set_title(f'Clasificación de postura — {pct_mal:.1f}% del tiempo encorvado', fontweight='bold')
plt.tight_layout()
plt.savefig('grafica_Postura.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7. Gráfica 5: Panel resumen (4 subplots) ──────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(13, 14), sharex=True)
fig.suptitle('VitalNode — Resumen completo de sesión', fontsize=14, fontweight='bold', y=1.01)

# BPM
axes[0].plot(df['Time_s'], df['BPM'], color=C_BPM, lw=1.6)
axes[0].axhline(df['BPM'].mean(), color=C_BPM, ls='--', alpha=0.5)
axes[0].set_ylabel('BPM')
axes[0].set_title('Frecuencia cardíaca')

# HRV
axes[1].fill_between(df['Time_s'], df['HRV_Index'], alpha=0.25, color=C_HRV)
axes[1].plot(df['Time_s'], df['HRV_Index'], color=C_HRV, lw=1.4)
axes[1].set_ylabel('HRV')
axes[1].set_title('Variabilidad cardíaca (HRV)')

# Respiración
axes[2].plot(df['Time_s'], df['Respiration_Raw'], color=C_RESP, lw=1.2)
axes[2].set_ylabel('FSR raw')
axes[2].set_title('Señal respiratoria')

# Postura
for i in range(len(df) - 1):
    color = C_POSTURE if df['Posture_num'].iloc[i] == 1 else C_HRV
    axes[3].axvspan(df['Time_s'].iloc[i], df['Time_s'].iloc[i+1],
                    alpha=0.6, color=color, lw=0)
axes[3].set_yticks([])
axes[3].set_xlabel('Tiempo (s)')
axes[3].set_title('Postura (verde=recta · naranja=encorvada)')

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('grafica_Resumen.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Panel guardado como grafica_Resumen.png')

In [ ]:
# ── 8. Tabla de estadísticas resumen ─────────────────────────────────────────
stats = pd.DataFrame({
    'Variable':  ['BPM', 'HRV_Index', 'Respiration_Raw'],
    'Media':     [df['BPM'].mean(), df['HRV_Index'].mean(), df['Respiration_Raw'].mean()],
    'Mín':       [df['BPM'].min(),  df['HRV_Index'].min(),  df['Respiration_Raw'].min()],
    'Máx':       [df['BPM'].max(),  df['HRV_Index'].max(),  df['Respiration_Raw'].max()],
    'Std':       [df['BPM'].std(),  df['HRV_Index'].std(),  df['Respiration_Raw'].std()],
}).round(2)

pct_encorvado = df['Posture_num'].mean() * 100
print(f'\n📋 Estadísticas de la sesión ({df["Time_s"].max():.0f}s):')
print(stats.to_string(index=False))
print(f'\n🧍 Postura encorvada: {pct_encorvado:.1f}% del tiempo')
print(f'🟢 Postura correcta:  {100 - pct_encorvado:.1f}% del tiempo')

In [ ]:
# ── 9. Descargar todas las gráficas generadas ─────────────────────────────────
from google.colab import files
import os

graficas = ['grafica_BPM.png', 'grafica_HRV.png',
            'grafica_Respiracion.png', 'grafica_Postura.png', 'grafica_Resumen.png']

for g in graficas:
    if os.path.exists(g):
        files.download(g)
        print(f'⬇️  Descargando {g}')
print('✅ Listo')